# RAILGUN+ — Notebook 2: Evaluate (Greedy vs PIBT-Corrected)

The core experiment: SAME trained model, two rollouts, across agent counts. The gap is your result.

## 1. Mount, clone, install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT='/content/drive/MyDrive/railgun-plus'
CKPT_DIR=f'{DRIVE_ROOT}/checkpoints'
REPO_URL='https://github.com/YOUR_USERNAME/railgun-plus.git'  # <-- EDIT
%cd /content
![ -d railgun-plus ] && (cd railgun-plus && git pull) || git clone $REPO_URL
%cd /content/railgun-plus
!pip install -q pogema pogema-toolbox pyyaml tqdm
sys.path.insert(0,'/content/railgun-plus/src')

## 2. Load trained model

In [ ]:
import torch
from railgun_plus.models import RailgunUNet
device='cuda' if torch.cuda.is_available() else 'cpu'
model=RailgunUNet(6,5,base=64).to(device)
ckpt=torch.load(f'{CKPT_DIR}/latest.pt',map_location=device)
model.load_state_dict(ckpt['model']);model.eval()
print('epoch',ckpt['epoch'])

## 3. Build held-out test sets across agent counts

In [ ]:
from railgun_plus.data.generate import generate_with_pogema
AGENT_COUNTS=[16,32,64,96,128]
N_PER=20
test_sets={}
for k in AGENT_COUNTS:
    test_sets[k]=generate_with_pogema(N_PER,32,0.2,k,seed=1000+k)
    print(k,'agents:',len(test_sets[k]),'instances')

## 4. Run both rollouts

In [ ]:
from railgun_plus.solvers.corrector import greedy_rollout, corrected_rollout
from railgun_plus.eval.metrics import soc, makespan, all_reached, summarize

def evaluate(fn, insts):
    res=[]
    for inst in insts:
        ok,paths=fn(model,inst,device=device)
        res.append({'success':all_reached(paths,inst.goals),
                    'soc':soc(paths,inst.goals),
                    'makespan':makespan(paths,inst.goals)})
    return summarize(res)

rows=[]
for k in AGENT_COUNTS:
    g=evaluate(greedy_rollout,test_sets[k])
    c=evaluate(corrected_rollout,test_sets[k])
    rows.append((k,g['csr'],c['csr'],g['avg_soc_solved'],c['avg_soc_solved']))
    print(f'agents={k:4d} greedy={g["csr"]:.2f} corrected={c["csr"]:.2f}')

## 5. Headline plot: CSR vs agents

In [ ]:
import matplotlib.pyplot as plt
ks=[r[0] for r in rows]
plt.figure(figsize=(7,5))
plt.plot(ks,[r[1] for r in rows],'o--',label='Greedy (baseline RAILGUN)')
plt.plot(ks,[r[2] for r in rows],'s-',label='PIBT-corrected (ours)')
plt.xlabel('Number of agents');plt.ylabel('CSR');plt.ylim(0,1.05)
plt.title('Deadlock collapse vs PIBT correction');plt.legend();plt.grid(True)
plt.savefig(f'{DRIVE_ROOT}/csr_greedy_vs_corrected.png',dpi=150,bbox_inches='tight')
plt.show()

## 6. SoC comparison

In [ ]:
plt.figure(figsize=(7,5))
plt.plot(ks,[r[3] for r in rows],'o--',label='Greedy')
plt.plot(ks,[r[4] for r in rows],'s-',label='PIBT-corrected')
plt.xlabel('Number of agents');plt.ylabel('Avg SoC (solved)')
plt.title('Solution quality');plt.legend();plt.grid(True);plt.show()